## 🔧 Configuração inicial

### Instalação das bibliotecas

Execute a célula abaixo **uma única vez** para instalar todas as bibliotecas utilizadas neste notebook. Caso já as tenha instaladas em seu ambiente, você pode pular esta etapa.

In [1]:
# Instalação das bibliotecas necessárias (execute apenas uma vez)
!pip install pandas numpy matplotlib seaborn scipy scikit-learn imbalanced-learn -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Agora vamos importar as bibliotecas que utilizaremos ao longo de todo o notebook.

In [2]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Estatística
from scipy import stats

# Machine Learning (pré-processamento e seleção de features)
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, LabelEncoder,
    MinMaxScaler, StandardScaler
)
from sklearn.feature_selection import SelectKBest, f_classif, f_regression, RFE
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


---
# 📁 Módulo 2 — Carregamento e Exploração de Dados CSV

Nesta seção, carregamos o arquivo CSV e realizamos a primeira exploração do conjunto de dados: dimensões, colunas, tipos, estatísticas descritivas e cardinalidade.

In [3]:
# Carregamento do dataset
dataframe = pd.read_csv("all_spectra_combined.csv", sep=",", encoding="utf-8")

print(f"O dataset foi carregado com sucesso!")
print(f"Dimensões: {dataframe.shape[0]} linhas x {dataframe.shape[1]} colunas")

O dataset foi carregado com sucesso!
Dimensões: 3736 linhas x 36 colunas


### 2.1 Primeira inspeção visual

In [ ]:
dataframe.head()

In [ ]:
dataframe.tail()

### 2.2 Dimensões, colunas e tipos de dados

In [ ]:
print("Formato (linhas, colunas):", dataframe.shape)
print("\nNomes das colunas:")
print(dataframe.columns.tolist())
print("\nNomes das colunas:")

Isso aqui informa os dados de cada coluna, nesse caso são 36 delas, todas completamente preenchidas com dados do tipo float64.

In [ ]:
# Isso aqui informa os dados de cada coluna, nesse caso são 36 delas
# todas completamente preenchidas com dados do tipo float64
dataframe.info()

In [ ]:
# pegando os datatype do frame e contando os valores, nesse caso tem 36 float64
dataframe.dtypes.value_counts()

### 2.3 Estatísticas descritivas

`df.describe()` resume as variáveis **numéricas**. Para incluir também as **categóricas**, usamos `include="all"`.

In [ ]:
# como nessa base só tem numéricas, já funciona só assim
dataframe.describe()

In [ ]:
# o T se faz a matriz ser transversal
dataframe.describe().T.head(20)

### 2.4 Cardinalidade e distribuição de classes

A cardinalidade indica quantos valores únicos existem em cada coluna — importante para decidir estratégias de codificação (Módulo 5).

Fazendo a análise dos valores, dá pra ver que a enorme maioria são únicos, com um máximo de 3 repetidos por linha. Conferir esses poucos valores em passos futuros.

In [ ]:
cardinalidade = dataframe.nunique().sort_values(ascending=False)
cardinalidade

In [ ]:
# Distribuição de uma variável categórica de interesse
dataframe["M2_1"].value_counts()

Um valor repetido, "-0.013667", e todo o resto único

---
# 🧪 Módulo 3 — Qualidade dos Dados

Vamos investigar valores ausentes, duplicados e outros problemas comuns de qualidade.

### 3.1 Valores ausentes

Felizmente, nessa base não há nenhum valor ausente, então essa etapa podem ser ignorada.

### 3.2 Dados duplicados

In [ ]:
duplicados = dataframe.duplicated().sum()
print(f"Número de linhas totalmente duplicadas: {duplicados}")

duplicados_sem_wavenumber = dataframe.drop(columns=["wavenumber"]).duplicated().sum()
print(f"Número de linhas duplicadas (ignorando o wavenumber): {duplicados_sem_wavenumber}")

### 3.3 Valores inválidos / fora do domínio esperado

Uma boa prática é verificar se variáveis numéricas possuem valores fisicamente impossíveis (ex.: área negativa, ano de construção no futuro).

In [18]:
dataframe["M1_2"].describe()

count    3736.000000
mean        0.030800
std         0.060212
min        -0.136756
25%         0.009589
50%         0.049690
75%         0.068341
max         0.148568
Name: M1_2, dtype: float64

### 3.4 Tipos incorretos

Vamos verificar se alguma coluna que deveria ser numérica foi carregada como texto (o que pode acontecer quando há símbolos ou valores como `"NA"` misturados).

In [ ]:
# Colunas numéricas esperadas, mas carregadas como object, indicam possível problema de tipo
for col in dataframe.columns:
    if dataframe[col].dtype == "object":
        amostra = dataframe[col].dropna().unique()[:5]
        # nada a corrigir aqui automaticamente — apenas inspeção
        pass

print("Total de colunas do tipo object (texto):", (dataframe.dtypes == "object").sum())
print("Total de colunas numéricas:", dataframe.select_dtypes(include=np.number).shape[1])

### 3.5 Tratamento inicial de valores ausentes (exemplo)

Vamos aplicar as três estratégias clássicas em colunas diferentes, apenas para fins didáticos (o tratamento definitivo será consolidado no Módulo 4).

In [ ]:
# Exemplo: imputação pela mediana (variável numérica com poucos ausentes)
mediana_lotfrontage = dataframe["LotFrontage"].median()
print("Mediana de LotFrontage:", mediana_lotfrontage)

# Exemplo: imputação pela moda (variável categórica com poucos ausentes)
moda_mszoning = dataframe["MSZoning"].mode()[0]
print("Moda de MSZoning:", moda_mszoning)

# Exemplo: remoção de linhas (não recomendado aqui, apenas ilustrativo)
linhas_antes = dataframe.shape[0]
linhas_sem_zoning_ausente = dataframe.dropna(subset=["MSZoning"]).shape[0]
print(f"Linhas antes: {linhas_antes} | Linhas se removêssemos ausentes de MSZoning: {linhas_sem_zoning_ausente}")

---
# 🧹 Módulo 4 — Limpeza e Padronização de Dados

Nesta seção, tratamos de forma consolidada os principais problemas identificados no Módulo 3, criando uma versão limpa do dataset (`df_limpo`).

### 4.1 Limpeza de strings

Vamos padronizar espaços e capitalização das colunas de texto.

In [ ]:
dataframe_limpo = dataframe.copy()

colunas_texto = dataframe_limpo.select_dtypes(include="object").columns

for col in colunas_texto:
    dataframe_limpo[col] = dataframe_limpo[col].astype(str).str.strip()

print("Espaços removidos das colunas de texto.")

### 4.2 Tratamento de valores ausentes por significado

Para as colunas em que `NA` representa **ausência do item** (e não dado faltante), substituímos por uma categoria explícita `"Nao_Possui"`. Para colunas numéricas relacionadas, substituímos por 0.

In [ ]:
# Colunas categóricas em que NA significa "não possui o item"
colunas_na_significativo = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature", "MasVnrType"
]

for col in colunas_na_significativo:
    dataframe_limpo[col] = dataframe_limpo[col].replace("nan", np.nan)
    dataframe_limpo[col] = dataframe_limpo[col].fillna("Nao_Possui")

print("Valores ausentes com significado tratados como categoria explícita.")

In [ ]:
# Colunas numéricas associadas a itens que podem não existir (ex.: área de garagem = 0 se não há garagem)
colunas_num_zero = ["MasVnrArea", "GarageYrBlt", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF"]

for col in colunas_num_zero:
    dataframe_limpo[col] = dataframe_limpo[col].fillna(0)

print("Valores ausentes numéricos (itens inexistentes) preenchidos com 0.")

### 4.3 Tratamento de valores realmente ausentes

Para colunas em que o valor ausente representa, de fato, dado faltante (não um "não se aplica"), aplicamos mediana (numéricas) ou moda (categóricas).

In [ ]:
# Verificando o que ainda resta de ausente
ausentes_restantes = dataframe_limpo.isnull().sum()
ausentes_restantes = ausentes_restantes[ausentes_restantes > 0].sort_values(ascending=False)
ausentes_restantes

In [ ]:
for col in ausentes_restantes.index:
    if dataframe_limpo[col].dtype in [np.float64, np.int64]:
        valor = dataframe_limpo[col].median()
        dataframe_limpo[col] = dataframe_limpo[col].fillna(valor)
    else:
        valor = dataframe_limpo[col].mode()[0]
        dataframe_limpo[col] = dataframe_limpo[col].fillna(valor)

print("Valores ausentes remanescentes tratados com mediana/moda.")
print("Total de valores ausentes após o tratamento:", dataframe_limpo.isnull().sum().sum())

### 4.4 Padronização de categorias

Vamos garantir que valores textuais equivalentes estejam escritos de forma consistente (letras minúsculas, sem espaços extras).

In [ ]:
dataframe_limpo["Neighborhood"] = dataframe_limpo["Neighborhood"].str.strip().str.title()
dataframe_limpo["Neighborhood"].unique()[:10]

### 4.5 Conversão de tipos

A coluna `MSSubClass` representa, na verdade, um **código categórico** (classe do imóvel), embora esteja armazenada como número. Vamos convertê-la para string, evitando que seja tratada erroneamente como variável numérica contínua.

In [ ]:
dataframe_limpo["MSSubClass"] = dataframe_limpo["MSSubClass"].astype(str)
print(dataframe_limpo["MSSubClass"].dtype)
dataframe_limpo["MSSubClass"].value_counts().head()

> ✅ A partir daqui, utilizaremos `df_limpo` como base para os próximos módulos.

---
# 🏷️ Módulo 5 — Variáveis Categóricas

Vamos identificar variáveis nominais e ordinais e aplicar as técnicas de codificação apropriadas para cada uma.

### 5.1 Identificando variáveis nominais e ordinais

- **Nominal:** `Neighborhood` (bairro) — não existe ordem entre as categorias.
- **Ordinal:** `ExterQual` (qualidade do acabamento externo) — possui uma ordem lógica de qualidade: `Po < Fa < TA < Gd < Ex`.

In [ ]:
print("Categorias de Neighborhood (nominal):", dataframe_limpo["Neighborhood"].nunique())
print("Categorias de ExterQual (ordinal):", dataframe_limpo["ExterQual"].unique())

### 5.2 Ordinal Encoding

Para `ExterQual`, respeitamos a ordem lógica das categorias ao codificá-las.

In [ ]:
ordem_qualidade = ["Po", "Fa", "TA", "Gd", "Ex"]

encoder_ordinal = OrdinalEncoder(categories=[ordem_qualidade])
dataframe_limpo["ExterQual_encoded"] = encoder_ordinal.fit_transform(dataframe_limpo[["ExterQual"]])

dataframe_limpo[["ExterQual", "ExterQual_encoded"]].drop_duplicates().sort_values("ExterQual_encoded")

### 5.3 Label Encoding

Aplicado, por exemplo, a variáveis binárias como `CentralAir` (Sim/Não), em que a codificação numérica não introduz uma falsa relação de ordem.

In [ ]:
label_encoder = LabelEncoder()
dataframe_limpo["CentralAir_encoded"] = label_encoder.fit_transform(dataframe_limpo["CentralAir"])

dataframe_limpo[["CentralAir", "CentralAir_encoded"]].drop_duplicates()

### 5.4 One-Hot Encoding

Para variáveis **nominais** com poucas categorias, como `Street` (tipo de acesso à rua), o One-Hot Encoding é a técnica mais adequada, pois não introduz relação de ordem.

In [ ]:
dataframe_onehot = pd.get_dummies(dataframe_limpo, columns=["Street"], prefix="Street")
dataframe_onehot[[c for c in dataframe_onehot.columns if "Street" in c]].head()

### 5.5 Alta cardinalidade

`Neighborhood` possui muitas categorias distintas — aplicar One-Hot Encoding diretamente geraria dezenas de novas colunas.

In [ ]:
print(f"Neighborhood possui {dataframe_limpo['Neighborhood'].nunique()} categorias distintas.")

# Se aplicássemos One-Hot Encoding diretamente:
colunas_geradas = pd.get_dummies(dataframe_limpo["Neighborhood"]).shape[1]
print(f"One-Hot Encoding geraria {colunas_geradas} novas colunas — alta cardinalidade!")

Uma estratégia comum é agrupar categorias raras em `"Outros"` antes de aplicar o One-Hot Encoding.

In [ ]:
frequencia = dataframe_limpo["Neighborhood"].value_counts()
categorias_frequentes = frequencia[frequencia >= 30].index

dataframe_limpo["Neighborhood_agrupado"] = dataframe_limpo["Neighborhood"].apply(
    lambda x: x if x in categorias_frequentes else "Outros"
)

print(f"Categorias após agrupamento: {dataframe_limpo['Neighborhood_agrupado'].nunique()}")
dataframe_limpo["Neighborhood_agrupado"].value_counts()

---
# 🔢 Módulo 6 — Variáveis Numéricas

Vamos calcular as principais medidas de tendência central e dispersão para variáveis numéricas relevantes.

In [ ]:
variaveis_numericas = ["LotArea", "GrLivArea", "OverallQual", "YearBuilt", "TotalBsmtSF"]

resumo = pd.DataFrame({
    "media": dataframe_limpo[variaveis_numericas].mean(),
    "mediana": dataframe_limpo[variaveis_numericas].median(),
    "moda": dataframe_limpo[variaveis_numericas].mode().iloc[0],
    "variancia": dataframe_limpo[variaveis_numericas].var(),
    "desvio_padrao": dataframe_limpo[variaveis_numericas].std(),
    "amplitude": dataframe_limpo[variaveis_numericas].max() - dataframe_limpo[variaveis_numericas].min(),
})
resumo.round(2)

### 6.1 Quartis e percentis

In [ ]:
dataframe_limpo["GrLivArea"].quantile([0.25, 0.5, 0.75])

In [ ]:
# Percentis customizados
dataframe_limpo["GrLivArea"].quantile([0.05, 0.10, 0.90, 0.95])

### 6.2 Discreta vs. contínua

- `OverallQual` (nota de qualidade de 1 a 10) é uma variável **discreta**.
- `GrLivArea` (área construída em pés²) é uma variável **contínua**.

In [ ]:
print("Valores únicos de OverallQual (discreta):", sorted(dataframe_limpo['OverallQual'].unique()))
print("\nGrLivArea (contínua) — min:", dataframe_limpo['GrLivArea'].min(), "| max:", dataframe_limpo['GrLivArea'].max())

> 💡 **Interpretação:** compare a média e a mediana de `GrLivArea` — se forem muito diferentes, isso é um indício de assimetria na distribuição (veremos isso formalmente no Módulo 8).

In [ ]:
print("Média de GrLivArea:", dataframe_limpo["GrLivArea"].mean().round(2))
print("Mediana de GrLivArea:", dataframe_limpo["GrLivArea"].median())

---
# 📈 Módulo 7 — Outliers

Vamos detectar outliers na variável `GrLivArea` (área construída) usando os métodos IQR e Z-score, e discutir estratégias de tratamento.

### 7.1 Detecção visual com boxplot

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=dataframe_limpo["GrLivArea"], color="#2E75B6")
plt.title("Boxplot — Área Construída (GrLivArea)")
plt.xlabel("Área construída (pés²)")
plt.show()

### 7.2 Método IQR (Intervalo Interquartil)

In [ ]:
Q1 = dataframe_limpo["GrLivArea"].quantile(0.25)
Q3 = dataframe_limpo["GrLivArea"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_iqr = dataframe_limpo[(dataframe_limpo["GrLivArea"] < limite_inferior) | (dataframe_limpo["GrLivArea"] > limite_superior)]

print(f"Q1 = {Q1} | Q3 = {Q3} | IQR = {IQR}")
print(f"Limite inferior: {limite_inferior:.2f} | Limite superior: {limite_superior:.2f}")
print(f"Número de outliers detectados (IQR): {len(outliers_iqr)}")

### 7.3 Método Z-score

In [ ]:
dataframe_limpo["zscore_GrLivArea"] = np.abs(stats.zscore(dataframe_limpo["GrLivArea"]))
outliers_zscore = dataframe_limpo[dataframe_limpo["zscore_GrLivArea"] > 3]

print(f"Número de outliers detectados (Z-score > 3): {len(outliers_zscore)}")
outliers_zscore[["GrLivArea", "zscore_GrLivArea"]].sort_values("zscore_GrLivArea", ascending=False).head()

### 7.4 Discussão sobre o tratamento

Vamos observar os imóveis identificados como outliers: eles podem representar **casas legítimas, porém excepcionalmente grandes** — não necessariamente um erro de digitação.

In [ ]:
outliers_iqr[["GrLivArea", "OverallQual", "Neighborhood", "YearBuilt"]].sort_values("GrLivArea", ascending=False).head(10)

> ⚠️ **Nem todo outlier deve ser removido.** Neste caso, os imóveis com área muito grande costumam ter também nota de qualidade elevada — são mansões reais, não erros. A decisão de remover, limitar (capping) ou manter deve considerar o contexto do problema.

Como exemplo, aplicamos abaixo a técnica de **limitação (capping)**, apenas para fins ilustrativos:

In [ ]:
dataframe_capping = dataframe_limpo.copy()
dataframe_capping["GrLivArea_capped"] = dataframe_capping["GrLivArea"].clip(lower=limite_inferior, upper=limite_superior)

print("Máximo antes do capping:", dataframe_capping["GrLivArea"].max())
print("Máximo depois do capping:", dataframe_capping["GrLivArea_capped"].max())

---
# 📐 Módulo 8 — Análise Estatística Exploratória (EDA)

Vamos calcular um conjunto completo de medidas estatísticas — incluindo assimetria e curtose — para compreender melhor a distribuição das variáveis numéricas.

In [ ]:
def resumo_estatistico(serie):
    return pd.Series({
        "média": serie.mean(),
        "mediana": serie.median(),
        "desvio_padrão": serie.std(),
        "variância": serie.var(),
        "Q1": serie.quantile(0.25),
        "Q3": serie.quantile(0.75),
        "assimetria": serie.skew(),
        "curtose": serie.kurtosis(),
    })

resumo_estatistico(dataframe_limpo["GrLivArea"]).round(2)

In [ ]:
resumo_estatistico(dataframe_limpo["SalePrice"] if "SalePrice" in dataframe_limpo.columns else dataframe_limpo["GrLivArea"]).round(2)

### 8.1 Interpretando a assimetria (skewness)

- **Assimetria ≈ 0:** distribuição aproximadamente simétrica.
- **Assimetria > 0:** cauda mais longa à direita (valores altos mais dispersos).
- **Assimetria < 0:** cauda mais longa à esquerda.

In [ ]:
variaveis = ["GrLivArea", "LotArea", "OverallQual", "YearBuilt"]

tabela_assimetria = pd.DataFrame({
    "assimetria": dataframe_limpo[variaveis].skew(),
    "curtose": dataframe_limpo[variaveis].kurtosis()
}).round(2)

tabela_assimetria

> 💡 `LotArea` costuma apresentar assimetria positiva alta — poucos terrenos muito grandes distorcem a distribuição. Isso motivará a transformação logarítmica no Módulo 14.

---
# 📊 Módulo 9 — Histogramas e Distribuições

Vamos visualizar a distribuição de diferentes variáveis e identificar seus formatos (normal, assimétrica, bimodal, uniforme).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(dataframe_limpo["GrLivArea"], bins=30, kde=True, ax=axes[0, 0], color="#2E75B6")
axes[0, 0].set_title("GrLivArea — assimétrica à direita")

sns.histplot(dataframe_limpo["YearBuilt"], bins=30, kde=True, ax=axes[0, 1], color="#548235")
axes[0, 1].set_title("YearBuilt — distribuição do ano de construção")

sns.histplot(dataframe_limpo["OverallQual"], bins=10, kde=False, ax=axes[1, 0], color="#BF8F00")
axes[1, 0].set_title("OverallQual — variável discreta (aprox. normal)")

sns.histplot(dataframe_limpo["LotArea"], bins=40, kde=True, ax=axes[1, 1], color="#C00000")
axes[1, 1].set_title("LotArea — forte assimetria positiva")

plt.tight_layout()
plt.show()

### 9.1 Interpretação

- `GrLivArea` e `LotArea`: distribuições **assimétricas à direita** — a maioria dos imóveis tem área moderada, mas alguns poucos têm área muito grande.
- `YearBuilt`: apresenta múltiplos picos, sugerindo **ciclos de construção** ao longo das décadas (padrão próximo de multimodal).
- `OverallQual`: por ser discreta e concentrada entre 4 e 8, se aproxima de uma distribuição **aproximadamente normal**.

---
# 🎲 Módulo 10 — Probabilidade

Vamos aplicar os conceitos fundamentais de probabilidade a eventos definidos sobre o próprio dataset.

### 10.1 Espaço amostral e eventos

Considere o experimento: **sortear aleatoriamente um imóvel do dataset**. O espaço amostral é o conjunto de todos os 1.459 imóveis.

Vamos definir dois eventos:
- **A:** o imóvel possui ar-condicionado central (`CentralAir == "Y"`)
- **B:** o imóvel foi construído após o ano 2000 (`YearBuilt > 2000`)

In [ ]:
total_imoveis = len(dataframe_limpo)

evento_A = dataframe_limpo["CentralAir"] == "Y"
evento_B = dataframe_limpo["YearBuilt"] > 2000

P_A = evento_A.sum() / total_imoveis
P_B = evento_B.sum() / total_imoveis

print(f"P(A) = P(possui ar-condicionado central) = {P_A:.4f}")
print(f"P(B) = P(construído após 2000) = {P_B:.4f}")

### 10.2 Probabilidade conjunta P(A ∩ B)

In [ ]:
P_A_e_B = (evento_A & evento_B).sum() / total_imoveis
print(f"P(A ∩ B) = P(possui ar-condicionado E foi construído após 2000) = {P_A_e_B:.4f}")

### 10.3 Probabilidade condicional P(A | B)

"Dado que o imóvel foi construído após 2000, qual a probabilidade de possuir ar-condicionado central?"


In [ ]:
P_A_dado_B = (evento_A & evento_B).sum() / evento_B.sum()
print(f"P(A | B) = {P_A_dado_B:.4f}")
print(f"Compare com P(A) = {P_A:.4f} (probabilidade sem a condição)")

> 💡 Se `P(A | B)` for muito maior que `P(A)`, isso sugere que **imóveis mais novos têm maior chance de possuir ar-condicionado central** — um indício de associação entre as variáveis, que investigaremos formalmente no Módulo 11.

### 10.4 Verificando independência

Dois eventos são independentes quando `P(A ∩ B) = P(A) × P(B)`.

In [ ]:
produto_marginal = P_A * P_B

print(f"P(A) x P(B) = {produto_marginal:.4f}")
print(f"P(A ∩ B) observado = {P_A_e_B:.4f}")

if abs(produto_marginal - P_A_e_B) < 0.01:
    print("Os eventos parecem aproximadamente independentes.")
else:
    print("Os eventos NÃO parecem independentes — há indícios de associação.")

---
# 🔗 Módulo 11 — Probabilidade Conjunta e Tabelas de Contingência

Vamos formalizar a investigação de associação entre variáveis categóricas usando tabelas de contingência.

### 11.1 Construindo a tabela de contingência

Vamos investigar a relação entre `CentralAir` (ar-condicionado central) e uma versão categorizada do ano de construção.

In [ ]:
dataframe_limpo["Epoca_Construcao"] = pd.cut(
    dataframe_limpo["YearBuilt"],
    bins=[1870, 1950, 1980, 2000, 2011],
    labels=["Ate_1950", "1951_1980", "1981_2000", "Apos_2000"]
)

tabela_contingencia = pd.crosstab(dataframe_limpo["Epoca_Construcao"], dataframe_limpo["CentralAir"])
tabela_contingencia

### 11.2 Convertendo em probabilidades

In [ ]:
# Probabilidade conjunta
tabela_conjunta = pd.crosstab(dataframe_limpo["Epoca_Construcao"], dataframe_limpo["CentralAir"], normalize="all")
print("Probabilidade conjunta P(Época ∩ CentralAir):")
tabela_conjunta.round(3)

In [ ]:
# Probabilidade condicional: P(CentralAir | Época)
tabela_condicional = pd.crosstab(dataframe_limpo["Epoca_Construcao"], dataframe_limpo["CentralAir"], normalize="index")
print("Probabilidade condicional P(CentralAir | Época):")
tabela_condicional.round(3)

### 11.3 Probabilidades marginais

In [ ]:
marginais_epoca = tabela_conjunta.sum(axis=1)
marginais_central_air = tabela_conjunta.sum(axis=0)

print("Probabilidade marginal por época:")
print(marginais_epoca.round(3))
print("\nProbabilidade marginal por CentralAir:")
print(marginais_central_air.round(3))

### 11.4 Visualizando a associação

In [ ]:
tabela_condicional.plot(kind="bar", stacked=True, figsize=(9, 5), color=["#C00000", "#2E75B6"])
plt.title("Proporção de imóveis com/sem ar-condicionado central, por época de construção")
plt.ylabel("Proporção")
plt.xlabel("Época de construção")
plt.legend(title="CentralAir", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> 💡 **Interpretação:** observe como a proporção de imóveis com ar-condicionado central cresce nas construções mais recentes — um forte indício de associação entre as duas variáveis, confirmando a intuição do Módulo 10.

---
# 🔄 Módulo 12 — Correlação

Vamos investigar a correlação entre variáveis numéricas, usando os coeficientes de Pearson e Spearman.

### 12.1 Matriz de correlação (Pearson)

In [ ]:
variaveis_interesse = [
    "LotArea", "OverallQual", "OverallCond", "YearBuilt", "TotalBsmtSF",
    "GrLivArea", "FullBath", "GarageCars", "GarageArea"
]

matriz_corr = dataframe_limpo[variaveis_interesse].corr(method="pearson")
matriz_corr.round(2)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, cmap="coolwarm", fmt=".2f", center=0, linewidths=0.5)
plt.title("Matriz de Correlação de Pearson")
plt.tight_layout()
plt.show()

### 12.2 Correlação de Spearman

Mais robusta a outliers e capaz de capturar relações monotônicas não necessariamente lineares.

In [ ]:
matriz_corr_spearman = dataframe_limpo[variaveis_interesse].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr_spearman, annot=True, cmap="coolwarm", fmt=".2f", center=0, linewidths=0.5)
plt.title("Matriz de Correlação de Spearman")
plt.tight_layout()
plt.show()

### 12.3 Identificando as correlações mais fortes

In [ ]:
corr_pares = matriz_corr.unstack().sort_values(ascending=False)
corr_pares = corr_pares[corr_pares < 1.0]  # remove a diagonal (correlação de uma variável com ela mesma)
corr_pares.drop_duplicates().head(10)

> ⚠️ **Correlação não implica causalidade.** `GarageCars` e `GarageArea` são fortemente correlacionadas — o que é esperado, já que garagens maiores comportam mais carros. Mas isso não significa que uma "causa" a outra: ambas refletem o mesmo atributo estrutural do imóvel (tamanho da garagem).

---
# 🖼️ Módulo 13 — Visualização de Dados

Vamos aplicar os principais tipos de gráficos apresentados na apostila para explorar visualmente o dataset.

### 13.1 Histograma

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(dataframe_limpo["GrLivArea"], bins=30, kde=True, color="#2E75B6")
plt.title("Histograma — Área construída (GrLivArea)")
plt.xlabel("Área construída (pés²)")
plt.show()

### 13.2 Boxplot (comparando grupos)

In [ ]:
plt.figure(figsize=(10, 6))
ordem = sorted(dataframe_limpo["OverallQual"].unique())
sns.boxplot(x="OverallQual", y="GrLivArea", data=dataframe_limpo, order=ordem, palette="Blues")
plt.title("Área construída por nota de qualidade geral (OverallQual)")
plt.xlabel("Qualidade geral (1-10)")
plt.ylabel("Área construída (pés²)")
plt.show()

### 13.3 Gráfico de dispersão (scatter plot)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x="GrLivArea", y="TotalBsmtSF", hue="OverallQual", data=dataframe_limpo, palette="viridis", alpha=0.7)
plt.title("Relação entre área construída e área do porão")
plt.xlabel("Área construída (pés²)")
plt.ylabel("Área do porão (pés²)")
plt.legend(title="Qualidade", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### 13.4 Gráfico de barras

In [ ]:
plt.figure(figsize=(11, 5))
dataframe_limpo["Neighborhood_agrupado"].value_counts().plot(kind="bar", color="#548235")
plt.title("Número de imóveis por bairro (agrupado)")
plt.xlabel("Bairro")
plt.ylabel("Quantidade de imóveis")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

### 13.5 Gráfico de linhas (tendência ao longo do tempo)

In [ ]:
imoveis_por_ano = dataframe_limpo.groupby("YearBuilt").size()

plt.figure(figsize=(11, 5))
imoveis_por_ano.plot(kind="line", color="#C00000")
plt.title("Número de imóveis construídos por ano")
plt.xlabel("Ano de construção")
plt.ylabel("Quantidade de imóveis")
plt.show()

### 13.6 Heatmap

Já construído no Módulo 12 (matriz de correlação) — reforça como o heatmap é a ferramenta ideal para visualizar relações entre múltiplas variáveis numéricas de uma só vez.

### 13.7 Pairplot

In [ ]:
subset_pairplot = dataframe_limpo[["GrLivArea", "TotalBsmtSF", "OverallQual", "YearBuilt"]].sample(300, random_state=42)

sns.pairplot(subset_pairplot, diag_kind="kde", plot_kws={"alpha": 0.5, "color": "#2E75B6"})
plt.suptitle("Pairplot de variáveis numéricas selecionadas", y=1.02)
plt.show()

> 💡 Usamos uma amostra de 300 linhas para o pairplot por questões de desempenho — com datasets maiores, gerar o gráfico completo pode ser lento.

---
# 🛠️ Módulo 14 — Engenharia de Features

Vamos criar novas variáveis (features) a partir das colunas existentes, aplicando as técnicas apresentadas na apostila.

### 14.1 Discretização (binning)

Transformando `OverallQual` (nota numérica) em uma faixa categórica.

In [ ]:
dataframe_limpo["Faixa_Qualidade"] = pd.cut(
    dataframe_limpo["OverallQual"],
    bins=[0, 3, 6, 8, 10],
    labels=["Baixa", "Media", "Boa", "Excelente"]
)

dataframe_limpo[["OverallQual", "Faixa_Qualidade"]].drop_duplicates().sort_values("OverallQual")

### 14.2 Features temporais

Vamos criar variáveis derivadas de `YearBuilt` e `YrSold`.

In [ ]:
dataframe_limpo["Idade_Imovel"] = dataframe_limpo["YrSold"] - dataframe_limpo["YearBuilt"]
dataframe_limpo["Foi_Reformado"] = (dataframe_limpo["YearRemodAdd"] != dataframe_limpo["YearBuilt"]).astype(int)
dataframe_limpo["Anos_Desde_Reforma"] = dataframe_limpo["YrSold"] - dataframe_limpo["YearRemodAdd"]

dataframe_limpo[["YearBuilt", "YrSold", "Idade_Imovel", "Foi_Reformado", "Anos_Desde_Reforma"]].head()

### 14.3 Transformação logarítmica

Reduz a assimetria de variáveis com distribuição enviesada, como `LotArea` (vista no Módulo 8/9).

In [ ]:
dataframe_limpo["LotArea_log"] = np.log1p(dataframe_limpo["LotArea"])  # log1p evita problemas com valores zero

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(dataframe_limpo["LotArea"], bins=40, ax=axes[0], color="#C00000")
axes[0].set_title(f"LotArea original (assimetria = {dataframe_limpo['LotArea'].skew():.2f})")

sns.histplot(dataframe_limpo["LotArea_log"], bins=40, ax=axes[1], color="#2E75B6")
axes[1].set_title(f"LotArea (log) (assimetria = {dataframe_limpo['LotArea_log'].skew():.2f})")

plt.tight_layout()
plt.show()

### 14.4 Razões e combinações entre variáveis

In [ ]:
dataframe_limpo["Area_Total"] = dataframe_limpo["GrLivArea"] + dataframe_limpo["TotalBsmtSF"]
dataframe_limpo["Banheiros_Totais"] = dataframe_limpo["FullBath"] + 0.5 * dataframe_limpo["HalfBath"] + \
                                 dataframe_limpo["BsmtFullBath"] + 0.5 * dataframe_limpo["BsmtHalfBath"]
dataframe_limpo["Razao_Garagem_Area"] = dataframe_limpo["GarageArea"] / dataframe_limpo["Area_Total"].replace(0, np.nan)

dataframe_limpo[["GrLivArea", "TotalBsmtSF", "Area_Total", "Banheiros_Totais", "Razao_Garagem_Area"]].head()

### 14.5 Features estatísticas por grupo (agregação)

Criando a média de área construída por bairro como uma nova feature — útil para capturar o "padrão" de cada região.

In [ ]:
media_area_bairro = dataframe_limpo.groupby("Neighborhood")["GrLivArea"].transform("mean")
dataframe_limpo["Media_Area_Bairro"] = media_area_bairro

dataframe_limpo[["Neighborhood", "GrLivArea", "Media_Area_Bairro"]].head()

### 14.6 Feature específica do domínio

In [ ]:
# Proporção da área construída em relação à área do terreno — indica o "aproveitamento" do lote
dataframe_limpo["Aproveitamento_Lote"] = dataframe_limpo["Area_Total"] / dataframe_limpo["LotArea"]

dataframe_limpo[["Area_Total", "LotArea", "Aproveitamento_Lote"]].describe().round(2)

---
# ⚖️ Módulo 15 — Normalização e Padronização

Vamos comparar as duas técnicas de escalonamento apresentadas na apostila.

In [ ]:
colunas_para_escalar = ["GrLivArea", "LotArea", "Area_Total", "Idade_Imovel"]

dataframe_escala = dataframe_limpo[colunas_para_escalar].copy()
dataframe_escala.describe().round(2)

### 15.1 Min-Max Scaling

In [ ]:
min_max_scaler = MinMaxScaler()
dataframe_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(dataframe_escala),
    columns=[f"{col}_minmax" for col in colunas_para_escalar]
)
dataframe_minmax.describe().round(3)

### 15.2 Standardization (StandardScaler)

In [ ]:
standard_scaler = StandardScaler()
dataframe_standard = pd.DataFrame(
    standard_scaler.fit_transform(dataframe_escala),
    columns=[f"{col}_standard" for col in colunas_para_escalar]
)
dataframe_standard.describe().round(3)

### 15.3 Comparação visual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(dataframe_escala["GrLivArea"], bins=30, ax=axes[0], color="#8C8C8C")
axes[0].set_title("Original")

sns.histplot(dataframe_minmax["GrLivArea_minmax"], bins=30, ax=axes[1], color="#2E75B6")
axes[1].set_title("Min-Max Scaling (0 a 1)")

sns.histplot(dataframe_standard["GrLivArea_standard"], bins=30, ax=axes[2], color="#548235")
axes[2].set_title("Standardization (média 0, DP 1)")

plt.tight_layout()
plt.show()

> 💡 **Observação importante:** note que a **forma** da distribuição (a assimetria) permanece a mesma nas três versões — o escalonamento muda apenas a escala dos valores, não a distribuição relativa dos dados. Por isso, variáveis assimétricas costumam ser primeiro transformadas (ex.: log, Módulo 14) e só depois escalonadas.

### 15.4 Regra de ouro: fit apenas no treino

Em um projeto real, o `fit` do scaler deve ser feito **apenas com os dados de treinamento**, e o `transform` aplicado depois aos dados de validação/teste — evitando vazamento de dados (*data leakage*).

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste = train_test_split(dataframe_escala, test_size=0.2, random_state=42)

scaler_correto = StandardScaler()
scaler_correto.fit(X_treino)  # ajusta apenas com dados de treino

X_treino_escalado = scaler_correto.transform(X_treino)
X_teste_escalado = scaler_correto.transform(X_teste)  # aplica os mesmos parâmetros ao teste

print("Scaler ajustado corretamente, sem vazamento de dados.")
print("Média do treino (usada no scaler):", X_treino["GrLivArea"].mean().round(2))

---
# 🎯 Módulo 16 — Seleção de Features

Vamos aplicar métodos de seleção de features para identificar as variáveis mais relevantes para prever a qualidade geral do imóvel (`OverallQual`), usada aqui como variável-alvo ilustrativa.

### 16.1 Preparando a base para seleção de features

In [ ]:
features_candidatas = [
    "LotArea", "YearBuilt", "TotalBsmtSF", "GrLivArea", "FullBath",
    "GarageCars", "GarageArea", "Idade_Imovel", "Area_Total", "Banheiros_Totais"
]

X = dataframe_limpo[features_candidatas].fillna(0)
y = dataframe_limpo["OverallQual"]

X.head()

### 16.2 Análise de correlação com a variável-alvo

In [ ]:
correlacao_com_alvo = X.assign(OverallQual=y).corr()["OverallQual"].drop("OverallQual").sort_values(ascending=False)
correlacao_com_alvo

### 16.3 Multicolinearidade entre features

Antes de selecionar, vamos identificar features redundantes entre si (alta correlação mútua).

In [ ]:
matriz_corr_features = X.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(matriz_corr_features, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlação entre as features candidatas (multicolinearidade)")
plt.tight_layout()
plt.show()

In [ ]:
# Identificando pares com correlação muito alta (> 0.8) — candidatos a redundância
pares_redundantes = matriz_corr_features.unstack()
pares_redundantes = pares_redundantes[(pares_redundantes.abs() > 0.8) & (pares_redundantes.abs() < 1.0)]
pares_redundantes.drop_duplicates()

> 💡 `Area_Total` foi construída a partir de `GrLivArea` e `TotalBsmtSF` (Módulo 14) — por isso apresenta alta correlação com ambas. Manter as três simultaneamente no modelo pode ser redundante.

### 16.4 SelectKBest

In [ ]:
selecionador = SelectKBest(score_func=f_regression, k=5)
selecionador.fit(X, y)

scores = pd.Series(selecionador.scores_, index=X.columns).sort_values(ascending=False)
print("Ranking de importância (F-score):")
print(scores)

features_selecionadas_kbest = X.columns[selecionador.get_support()]
print("\nFeatures selecionadas pelo SelectKBest:", list(features_selecionadas_kbest))

### 16.5 RFE (Recursive Feature Elimination)

In [ ]:
modelo_base = LinearRegression()
rfe = RFE(modelo_base, n_features_to_select=5)
rfe.fit(X, y)

features_selecionadas_rfe = X.columns[rfe.support_]
print("Features selecionadas pelo RFE:", list(features_selecionadas_rfe))

### 16.6 Importância de features via Random Forest

In [ ]:
modelo_rf = RandomForestRegressor(n_estimators=200, random_state=42)
modelo_rf.fit(X, y)

importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
importancias.plot(kind="barh", color="#2E75B6")
plt.title("Importância das features (Random Forest)")
plt.xlabel("Importância")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> 💡 **Conclusão do módulo:** os três métodos (correlação, SelectKBest e RFE/Random Forest) convergem, em geral, para features como `GrLivArea`, `Area_Total`, `TotalBsmtSF` e `GarageCars` como as mais relevantes — reforçando a ideia de que **mais features não significa necessariamente um modelo melhor**: o foco deve estar na relevância, não na quantidade.

---
# ⚠️ Módulo 17 — Dados Desbalanceados

Vamos utilizar a variável `Street` (tipo de acesso à via) como exemplo de classe fortemente desbalanceada, presente naturalmente neste dataset.

### 17.1 Identificando o desbalanceamento

In [ ]:
contagem_street = dataframe_limpo["Street"].value_counts()
proporcao_street = dataframe_limpo["Street"].value_counts(normalize=True) * 100

print(contagem_street)
print("\nProporção (%):")
print(proporcao_street.round(2))

In [ ]:
plt.figure(figsize=(6, 5))
contagem_street.plot(kind="bar", color=["#2E75B6", "#C00000"])
plt.title("Distribuição da variável Street (fortemente desbalanceada)")
plt.ylabel("Quantidade de imóveis")
plt.xticks(rotation=0)
plt.show()

> 💡 Apenas **6 imóveis** possuem acesso do tipo `Grvl` (cascalho), contra **1.453** do tipo `Pave` (pavimentado) — um desbalanceamento extremo (~99,6% vs. ~0,4%), similar ao exemplo de "1% falha / 99% normal" apresentado na apostila.

### 17.2 Por que a acurácia pode enganar

Se construíssemos um classificador ingênuo que **sempre prevê `"Pave"`**, qual seria sua acurácia?

In [ ]:
acuracia_ingenua = (dataframe_limpo["Street"] == "Pave").mean()
print(f"Acurácia de um modelo que sempre prevê 'Pave': {acuracia_ingenua:.4f} ({acuracia_ingenua*100:.2f}%)")
print("Apesar da acurácia altíssima, esse modelo NUNCA identificaria corretamente um imóvel com acesso 'Grvl'.")

### 17.3 Simulando undersampling e oversampling

Vamos ilustrar conceitualmente as duas técnicas, criando versões balanceadas do dataset (apenas com as colunas numéricas, para simplificar).

In [ ]:
colunas_exemplo = ["LotArea", "OverallQual", "GrLivArea", "Street"]
dataframe_desbalanceado = dataframe_limpo[colunas_exemplo].dropna()

classe_majoritaria = dataframe_desbalanceado[dataframe_desbalanceado["Street"] == "Pave"]
classe_minoritaria = dataframe_desbalanceado[dataframe_desbalanceado["Street"] == "Grvl"]

print(f"Classe majoritária (Pave): {len(classe_majoritaria)} registros")
print(f"Classe minoritária (Grvl): {len(classe_minoritaria)} registros")

In [ ]:
# Undersampling: reduz a classe majoritária ao tamanho da minoritária
undersample_majoritaria = classe_majoritaria.sample(n=len(classe_minoritaria), random_state=42)
dataframe_undersampled = pd.concat([undersample_majoritaria, classe_minoritaria])

print("Distribuição após undersampling:")
print(dataframe_undersampled["Street"].value_counts())

In [ ]:
# Oversampling: replica a classe minoritária até o tamanho da majoritária
oversample_minoritaria = classe_minoritaria.sample(n=len(classe_majoritaria), replace=True, random_state=42)
dataframe_oversampled = pd.concat([classe_majoritaria, oversample_minoritaria])

print("Distribuição após oversampling:")
print(dataframe_oversampled["Street"].value_counts())

### 17.4 SMOTE (conceitual)

O SMOTE gera exemplos **sintéticos** da classe minoritária, interpolando entre observações existentes (em vez de apenas duplicá-las, como no oversampling simples). Sua implementação prática requer a biblioteca `imbalanced-learn`:

```python
# pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=2)
X_resampled, y_resampled = smote.fit_resample(X, y)
```

> ⚠️ Com apenas 6 registros na classe minoritária, o SMOTE tem um número muito limitado de vizinhos para interpolar — nesse caso extremo, o undersampling ou a coleta de mais dados costumam ser alternativas mais adequadas.

### 17.5 Métricas apropriadas para dados desbalanceados

Em vez de depender apenas da acurácia, deve-se avaliar o modelo com métricas como **precisão**, **revocação (recall)**, **F1-score** e **AUC-ROC**, que consideram o desempenho especificamente na classe minoritária.

In [ ]:
from sklearn.metrics import classification_report

# Exemplo ilustrativo: um classificador ingênuo que sempre prevê a classe majoritária
y_real = dataframe_desbalanceado["Street"]
y_previsto_ingenuo = ["Pave"] * len(y_real)

print(classification_report(y_real, y_previsto_ingenuo, zero_division=0))

> 💡 **Observação final:** repare que, embora a acurácia global seja altíssima, a **precisão, revocação e F1-score da classe `Grvl`** são todas iguais a zero — o modelo ingênuo é completamente incapaz de identificar a classe minoritária. Este é exatamente o problema que as técnicas de balanceamento (undersampling, oversampling, SMOTE) e as métricas apropriadas buscam evidenciar e mitigar.

---
# ✅ Conclusão

Neste notebook, percorremos na prática os Módulos 2 a 17 da apostila *Engenharia e Preparação de Dados*, aplicando cada técnica ao dataset real de imóveis (`dados_imoveis.csv`):

- Carregamos e exploramos os dados brutos (Módulo 2);
- Diagnosticamos problemas de qualidade (Módulo 3) e os corrigimos (Módulo 4);
- Codificamos variáveis categóricas (Módulo 5) e analisamos variáveis numéricas (Módulo 6);
- Detectamos outliers (Módulo 7) e aprofundamos a análise estatística (Módulo 8);
- Visualizamos distribuições (Módulo 9) e aplicamos conceitos de probabilidade (Módulos 10 e 11);
- Investigamos correlações (Módulo 12) e construímos visualizações diversas (Módulo 13);
- Criamos novas features (Módulo 14) e as normalizamos (Módulo 15);
- Selecionamos as features mais relevantes (Módulo 16) e discutimos o desafio de classes desbalanceadas (Módulo 17).

> 🎯 **Próximos passos sugeridos:** aplicar os Módulos 21 e 22 da apostila (Separação entre Treinamento/Validação/Teste e Pipeline Completo de Pré-processamento) para consolidar tudo o que foi feito aqui em um pipeline único, reprodutível e pronto para alimentar um modelo de Machine Learning.
